In [1]:
import requests
import json
import pandas as pd

In [2]:
path_in = "C:/IMAGE/Foliage/city_Bangalore/"
path_out = "C:/IMAGE/Foliage/image_api/subgrids/"

file_in = "SUBGRID_DATA_IMAGEBASED_S4_BUILD+FOLIAGE_BLR.txt"
file_out = "SUBGRID_DATA_IMAGEBASED_BANGALORE.txt"

In [ ]:
df_merged = pd.read_csv(path_in+file_in,sep='\t')
df_merged.head()

In [ ]:
# AMSL Calculation

def get_resource(r):
    resourceSets = json.loads(r.text)
    try:
        for resource in resourceSets['resourceSets']:
            return [rsc for rsc in resource['resources']]
    except KeyError:
        try:
            resourcesSets = resourceSets['Response']['ResourceSets']
            resourceSet = resourcesSets['ResourceSet']
            if isinstance(resourceSet, dict):
                return resourceSet['Resources']
            elif isinstance(resourceSet, list):
                return [rsc for rsc in resourceSet['Resources']]
        except KeyError:
            print(KeyError)

def get_elevation(lat, lon):
    key = 'Ap2jFsGK3XZxoJYoaoE8pMLEk2B4x8OqgkolcqfDPwN-24_V9Vd__m5YOCyXPXch'
    point_query = f"{lat},{lon}"
    api_query = f'http://dev.virtualearth.net/REST/v1/Elevation/List?points={point_query}&heights=sealevel&key={key}'
    
    r = requests.get(api_query)
    if r.status_code != 200:
        raise r.raise_for_status()

    resource = get_resource(r)
    elevation = resource[0]['elevations'][0]

    return elevation

# Example usage:
#latitude = 35.66115  # Replace with the latitude of your point
#longitude = -115.717578  # Replace with the longitude of your point
list_elev = []
for i, _ in df_merged.iterrows():
    lat = df_merged["PT_LAT"][i]
    lon = df_merged["PT_LON"][i]

    elevation = get_elevation(lat, lon)
    list_elev.append(elevation)
    print(str(i),end="|")

#elevation = get_elevation(latitude, longitude)
#print(f"The AMSL elevation at the point ({latitude}, {longitude}) is {elevation} meters.")

In [ ]:
df_with_amsl = df_merged.copy()

In [ ]:
df_with_amsl["AMSL"] = pd.Series(data=list_elev)
df_with_amsl.head()

In [ ]:
#df_with_amsl.pd.to_csv(path_out+file_out,sep='\t',encoding='utf-8')